# 개발 환경 구축 따라하기

## 🎯 학습 목표

- `uv` 로 격리된 파이썬 환경을 만들고 의존성을 **재현 가능하게** 설치한다
- API 키를 `.env` 로 분리하는 이유를 설명하고 직접 설정한다

| 마커 | 뜻 |
|---|---|
| **▶ 함께 실행** | 강사가 실행한 셀을 **동일하게 실행**합니다. 코드는 그대로 두세요 |
| **🔧 미니실습 M#** | **값 하나를 바꾸거나 한 줄을 채웁니다.** 정답 셀이 바로 아래 있습니다 |

<br>

> ⚠️ **실행 결과의 수치는 환경이나 실행 시점에 따라 다를 수 있습니다.** LLM 은 비결정론적 특성이 있으므로, 고정된 규칙과 변동되는 결과를 비교해 보세요.

---

## 1. 왜 `uv` 인가 — pip + venv 가 아니라

`uv` 는 **가상환경 · 의존성 · 파이썬 버전**을 한 도구가 처리합니다.

| | pip + venv | uv |
|---|---|---|
| 파이썬 버전 | 별도 설치(pyenv 등) | **자동 설치** |
| 가상환경 | `python -m venv` 수동 | `uv sync` 가 알아서 |
| 잠금 | `requirements.txt` (해상도 불완전) | `uv.lock` (**정확한 재현**) |
| 속도 | — | 수~수십 배 |

우리에게 중요한 것은 속도가 아니라 **재현성**입니다.

> 라이브러리 버전이 다르면 교재의 실측 수치와 차이가 발생할 수 있습니다 —
> 실제로 이 과정의 `pyproject.toml` 은 주요 패키지를 `==` 로 **명시적으로 고정해** 두었습니다.

### 키는 코드에 넣지 않습니다

API 키를 노트북 셀에 직접 쓰면 **그 노트북은 공유할 수 없게 됩니다.** 캡처 한 장, 커밋 한 번이면 유출입니다.

```
프로젝트/
├── .env            ← 키는 여기에. git 에 올라가지 않는다
├── .env.example    ← 형식만 공유
└── notebooks/      ← 코드는 os.getenv() 로 읽기만 한다
```

---

## 2. 함께 실행 — 다섯 단계

**여기서부터 강사와 같이 진행합니다.** 화면을 보고 그대로 따라오세요.

> ### ▶ 함께 실행 — 설치 5단계
>
> **터미널에서 ①~③, 편집기에서 ④, 이 노트북에서 ⑤**를 진행합니다.
>
> ```
> ① uv 설치 확인          uv --version
> ② 배포 압축 해제 · 폴더 열기
> ③ uv sync               ← 가장 오래 걸립니다 (2~5분)
> ④ .env 생성 · OPENAI_API_KEY 입력
> ⑤ 아래 §4 검증 셀 실행  → ✅ 4줄
> ```
>
> 🔍 **③ 이 실행되는 동안** 강사는 §1 의 설명을 이어갑니다. 설치 완료를 기다리기보다 설명에 집중해 주시기 바랍니다.

### ① uv 설치 확인

`uv --version` 실행 시 버전 정보가 출력되면 다음 단계로 진행합니다.

| OS | 설치 |
|---|---|
| macOS | `brew install uv` |
| Windows | `winget install astral-sh.uv` |
| 공통(대안) | `curl -LsSf https://astral.sh/uv/install.sh \| sh` |

<br>

> ⚠️ 설치 직후 `uv: command not found` 에러가 발생하면 **터미널을 다시 실행하세요.** 환경 변수(PATH)가 아직 반영되지 않은 상태입니다.

---

### ② 배포 압축 해제 · 프로젝트 폴더 열기

배포된 실습 파일의 압축을 해제하고, VS Code 등의 편집기에서 프로젝트 폴더를 엽니다.

### ③ `uv sync` — 소요 시간이 가장 긴 단계입니다

프로젝트 폴더에서:

```bash
uv sync --extra build
```

`pyproject.toml` 의 핀 목록대로 `.venv/` 를 만듭니다. 회선에 따라 **2~5분**.

설치되는 주요 패키지 — 버전 고정 이유는 §1 에서 설명한 내용과 같습니다.

| 패키지 | 핀 |
|---|---|
| `langchain` | `==1.3.15` |
| `langchain-openai` | `==1.5.1` |
| `langgraph` | `==1.2.11` |
| `chromadb` | `==1.5.9` |
| `langfuse` | `>=4.15,<5` |


### ④ `.env` 만들기

프로젝트 루트의 `.env.example` 을 복사해 `.env` 로 이름을 바꾸고, 키를 채웁니다.

```bash
cp .env.example .env
```

```
OPENAI_API_KEY=sk-여기에-발급받은-키
```

---

## 3. 미니실습 M0 — 첫 호출


**바꿀 것**: 없습니다. **아래 셀을 그대로 실행**하세요.

**볼 것**: 응답 문자열이 출력되는가. 그리고 **입력·출력 토큰 수**가 함께 찍히는가.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), (
    ".env 에 OPENAI_API_KEY 가 없습니다. §2-④ 를 다시 보세요."
)

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
resp = llm.invoke("한 문장으로 자기소개를 해주세요.")

print(resp.content)
print()
# 토큰 사용량은 응답 메타데이터에 포함됩니다. 이 수치는 과정 전반에서 지속적으로 확인합니다.
print("입력 토큰:", resp.usage_metadata["input_tokens"])
print("출력 토큰:", resp.usage_metadata["output_tokens"])

---

## 4. 검증 셀 (⑤ 환경 검증)

In [ ]:
import sys, importlib.metadata as meta

def check(label, ok, detail=""):
    print(("✅" if ok else "❌"), label, detail)
    return ok

results = []
results.append(check("Python 3.12", sys.version_info[:2] == (3, 12),
                     f"(현재 {sys.version_info.major}.{sys.version_info.minor})"))

want = {"langchain": "1.3.15", "langgraph": "1.2.11", "langchain-openai": "1.5.1"}
got = {}
for name, pin in want.items():
    try:
        got[name] = meta.version(name)
    except meta.PackageNotFoundError:
        got[name] = "없음"
results.append(check("핵심 패키지 3종", all(got[n] == v for n, v in want.items()), str(got)))

results.append(check("OPENAI_API_KEY", bool(os.getenv("OPENAI_API_KEY")),
                     "(값은 출력하지 않습니다)"))

try:
    ping = llm.invoke("ping").content
    results.append(check("OpenAI 호출", bool(ping), f"→ {ping[:24]!r}"))
except Exception as error:
    results.append(check("OpenAI 호출", False, f"→ {type(error).__name__}: {error}"))

print()
print("=" * 52)
print(" 준비 완료 — 실습 진행이 가능합니다" if all(results)
      else " 미완료 — 추가 조치를 진행합니다")
print("=" * 52)

---

## 5. 문제 대응

| 증상 | 원인 | 조치 |
|---|---|---|
| `uv: command not found` | PATH 미반영 | **터미널을 다시 실행합니다.** 해결되지 않으면 설치 명령 재실행 |
| `AuthenticationError` | 키 앞뒤 공백 | `.env` 를 공백 **없이** `OPENAI_API_KEY=sk-...` 로 |
| `RateLimitError` | 공용 키에 동시 접속 | 30초 뒤 재실행. |
| 커널이 `.venv` 를 못 찾음 | 편집기가 다른 파이썬을 봄 | VS Code 우상단 커널 선택 → `.venv/bin/python` |

---

## 정리

- `uv sync` 로 **버전이 고정된** 환경을 구성했습니다 — 실습 환경의 일치성을 보장합니다
- API 키는 `.env` 로 분리하여 관리합니다. 노트북 코드에는 절대 직접 작성하지 않습니다
- 응답과 함께 반환되는 **입력·출력 토큰**은 과정 전반에서 지속적으로 추적하는 핵심 지표입니다